We first import all the necesary libraries. Some are not used, but included for possible future use. Note that the path should be pointing to your dataset.

In [ ]:
# SETUP libraries and file paths
from pathlib import Path
from pytact.data_reader import data_reader
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import json
import scipy

DATASET_PATH = Path("/Users/huubdejong/Documents/Education/UBC/M2PI_2025/Project/TacticianDataTemp/TacticianDataTemp/TacticianDataTemp/v15-stdlib-coq8.11/dataset").resolve()


We find a way to identify a tactic that's used in a proof. The "Done" classifier is used for two and three step probabilities, and indicates that no more tactics were applied. From the pytact documentation: 

"Sometimes a tactic cannot or should not be recorded. In those cases, it is marked as 'unknown'. This currently happens with tactics that are run as a result of the Proof with tac construct and it happens for tactics that are known to be unsafe like change_no_check, fix, cofix and more."

 In this case, the ProofStep.tactic field is set to None, and we treat it as an "Unknown" tactic. This occurs only in a fraction of the dataset.

In [ ]:
# helper to convert tactic into string
def classifyTactic(step):
    if step == None:
        return "Done"
    elif step.tactic == None:
        return "Unknown"
    else:
        return step.tactic.ident


Below is the code for our displacement "metric". A current placehord is the Jaccard similarity between the text representation of two proofstates $s1$ and $s2$:
$$
\operatorname{JaccardSimilarity}(s1,s2)=\frac{|\operatorname{words}(s1)\cap \operatorname{words}(s2)|}{|\operatorname{words}(s1)\cup \operatorname{words}(s2)|}.
$$

This is suboptimal for several reason, which essentially boil down to the fact that this kind of similarity ignores most structural properties of the proofstate. Some potential alternative ideas are in https://arxiv.org/pdf/2404.08817, and improving this part could be a great improvement in this project.

In [ ]:
def Jaccard_Similarity(phrase1, phrase2): 
    words_doc1 = set(phrase1.split()) 
    words_doc2 = set(phrase2.split())
    intersection = words_doc1.intersection(words_doc2)
    union = words_doc1.union(words_doc2)
    return float(len(intersection)) / len(union)
    

def proofStateDistance(state1,state2):
    return 1 - Jaccard_Similarity(state1.conclusion_text, state2.conclusion_text)

Next, we parse all the data we are interested in. The data will be stored in a nested dictionary indexed by Tactic classifiers. 

In [ ]:
rich = True

# These will be the main storages for our data
tacticData = {}
treeDepths = []
proofLengths = []
definitionsCounter = 0
prooflessCounter = 0

# this part parses the data and stores everything we might want to know about
with data_reader(DATASET_PATH) as data:
    for idx, (relative_path,dataset) in enumerate(data.items()):
        if rich: print("Parsing \"" + str(relative_path) + "\" for proofs...")
        for definition in dataset.definitions(spine_only=False): 
            definitionsCounter += 1
            if definition.proof == None:  
                prooflessCounter += 1
                continue

            # There is a proof, so we can begin processing the data
            proof = definition.proof
            G = nx.DiGraph() #This graph can also be used for visualization purposes

            # BEGIN processing data for this definition
            for i in range(len(proof)):
                # convert tactics 
                tactic1 = classifyTactic(proof[i])
                tactic2 = classifyTactic(proof[i+1] if i < len(proof) - 1 else None)
                tactic3 = classifyTactic(proof[i+2] if i < len(proof) - 2 else None)

                # insert field in main data
                if tactic1 not in tacticData:
                    tacticData.update({tactic1 : 
                                    {"text" : [proof[i].tactic.base_text if proof[i].tactic != None else "Unknown"],  
                                        "exact" : proof[i].tactic.exact if proof[i].tactic != None else False,
                                        "count": 0,
                                        "nextCount": {},
                                        "twoNextCount":{},
                                        "countPerFile": {relative_path.name: 0},
                                        "displacements": [] }})

                # Update tactic count 
                tacticData[tactic1]["count"] += 1

                # update tactic count per file
                if relative_path.name not in tacticData[tactic1]["countPerFile"]:
                    tacticData[tactic1]["countPerFile"].update({relative_path.name:0})    
                tacticData[tactic1]["countPerFile"][relative_path.name] += 1
                
                # update tactic text representation
                if proof[i].tactic != None and proof[i].tactic.base_text not in tacticData[tactic1]["text"]:
                    tacticData[tactic1]["text"].append(proof[i].tactic.base_text)
                
                # update tactic count doubles
                if tactic2 not in tacticData[tactic1]["nextCount"]:
                    tacticData[tactic1]["nextCount"].update({tactic2:0})
                tacticData[tactic1]["nextCount"][tactic2] += 1

                # update tactic count triples
                if tactic2 not in tacticData[tactic1]["twoNextCount"]:
                    tacticData[tactic1]["twoNextCount"].update({tactic2:{}})
                if tactic3 not in tacticData[tactic1]["twoNextCount"][tactic2]:
                    tacticData[tactic1]["twoNextCount"][tactic2].update({tactic3:0})
                tacticData[tactic1]["twoNextCount"][tactic2][tactic3] += 1

                # update proof graph and measure displacement
                for outcome in proof[i].outcomes:
                    for afterState in outcome.after:
                        G.add_edge(outcome.before.conclusion_text, afterState.conclusion_text)
                        tacticData[tactic1]["displacements"].append(proofStateDistance(outcome.before,afterState))
                        
            # count depth of tree and number of tactics used in the proof
            if nx.is_directed_acyclic_graph(G):
                treeDepths.append(nx.dag_longest_path_length(G)+1)
                proofLengths.append(len(proof))
                
            # END processing data for this definition

We quickly store all the data in a Json file, so we do not need to parse the dataset again.

In [ ]:
with open("TacticData.json", "w") as fp:
    json.dump(tacticData , fp, indent=4) 

Let's find out two things: How many tactics were used fewer than, say, 5 times? How many definition objects did not come with a proof?

In [ ]:
# print basic information
n = 5
print(str(sum([1 if (v['count'] <= n) else 0 for _,v in tacticData.items()])) + " out of " + str(len(tacticData.keys()))  + " tactics were used at most " + str(n) + " times.")
print("Total number of proofless definitions: " + str(prooflessCounter) + " out of " + str(definitionsCounter)) 


We'll also check if any hash collisions occured for the hash classifiers. We do this by checking if any classifier has more than one text representation associated with it. If it does, it may still be the same tactic, but called from a different file. For example, "elim" and "MX.elim" should, in theory, still be the same tactic, and we can ignore the hash collision.

In [ ]:
collided = False
for k,v in tacticData.items():
    if len(v['text']) > 1:
        collided = True
        print("The following tactics have hash " + str(k) + ":")
        print(v['text'])
if not collided:
    print("No tactichash collisions detected.")

Let's print the distribution of tactics by how often they are used.

In [ ]:
sns.kdeplot([v['count'] for v in tacticData.values()],log_scale=True)
plt.xlabel("Number of times tactic was used")
plt.savefig("Total_TacticCountDistribution.svg")
plt.show()

We'll also check how this distribution looks per file. This part only works on the "v15-stdlib-coq8.11" dataset, which has a relatively small number of files.

In [ ]:
data = {}
for tact, dict in tacticData.items():
  for file, count in dict['countPerFile'].items():
    if file not in data:
      data.update({file:{}})
    if str(tact) not in data[file]:
      data[file].update({str(tact): 0})
    data[file][str(tact)] += count

sns.kdeplot(data, log_scale=True, legend=False, warn_singular=False)
plt.savefig("PerFile_TacticCountDistribution.svg")
plt.title("Number of tactic uses per file")

We can also check how long proofs are.

In [ ]:
sns.histplot(proofLengths,log_scale=False)
plt.xlabel("length of the proof")
plt.savefig("ProofLengths.svg")
plt.show()

And how deep the proofstate tree graph is.

In [ ]:
sns.histplot(treeDepths,log_scale=False)
plt.xlabel("depth of the proof state tree")
plt.savefig("TreeDepths.svg")
plt.show()

The ratios of these two values concentrate around 1, which suggests that a vast majority of proofs is linear.

In [ ]:
ratios = [float(treeDepths[i])/proofLengths[i] for i in range(len(treeDepths))]
sns.histplot(ratios,log_scale=False)
plt.xlabel("Depth of the proof tree / Length of the proof")
plt.savefig("TreeDepthToProofLengthRatios.svg")
plt.show()

What if we pick a random tactic, $T1$? Or a pair of consecutive tactics, $(T1,T2)$? The entropies of these variables tell us something about how related they are.

In [ ]:
print("H(T1) = " + str(scipy.stats.entropy(pk=[v['count'] for v in tacticData.values()])))

In [ ]:
totalx = 0
out = 0
for k,v in tacticData.items():
    totalx += v['count']
    out += v['count'] * scipy.stats.entropy(pk = [p for p in v['nextCount'].values()])
conditionalEntropy = out / float(totalx)
print("H(T2|T1) = " + str(conditionalEntropy))

In [ ]:
totalDist = {}
for _, row in tacticData.items():
    for k,v in row['nextCount'].items():
        if k in totalDist:
            totalDist[k] += v
        else:
            totalDist.update({k:v})

secondStepEntropy = scipy.stats.entropy(pk=[p for p in totalDist.values()])
print("H(T2) = " + str(secondStepEntropy))
print("I(T1,T2) = H(T2) - H(T2|T1) = " + str(secondStepEntropy - conditionalEntropy))

We'll also look at the data of some tactics we deem interesting. On one hand, "intros", "destruct", and "easy" are very common. However, we conjecture that "induction", "rewrite", "exact", and "apply" are much more likely to be a key step in the proof. We collect their hashes, and some others, here, and pick our favourite.

For our choice, we display the distributions of the preceding and succeeding tactics, the two succeeding tactics, and the displacement as caused by the use of this tactic.

In [ ]:
tacticHash = 3192827940261208899 # intros: 2091839244048452363 or destruct: 9120222886189897970 or easy: 3705308508539391475
# induction: 7871868389188524885, rewrite: 3535549731613791139 , rewrite <-: 4237068880675071532, exact: -8499735706083503724	, apply: 3192827940261208899, 
# there are also lots of variations such as: rewrite _ in __: 9050438711924037266, rewrite <- in : 3579137785924911933, coinduction ltac: -8126988681580081290, many apply

data = tacticData[tacticHash]['nextCount']
totaly = 0
for v in data.values():
  totaly += v
labels = [tacticData[k]['text'][0] if (v/float(totaly) > 0.03 and k != "Done") else "Done" if (v/float(totaly) > 0.03 and k == "Done") else ''  for k,v in data.items()]
plt.pie(data.values(), labels=labels)
plt.title("Distribution of tactics used after " + str(tacticData[tacticHash]['text'][0]))
plt.savefig(str(tacticHash) + "_OneStepForward.svg")
plt.show()

In [ ]:
data = {}
for tact, dist in tacticData.items():
  if tacticHash in dist['nextCount']:
    data.update({tacticData[tact]['text'][0] : dist['nextCount'][tacticHash]})
total = 0
for v in data.values():
  total+= v
plt.pie(data.values(),labels=[key if data[key]/float(total) > 0.02 else '' for key in data.keys()])
plt.title("Distribution of tactics used before " + str(tacticData[tacticHash]['text'][0]))
plt.savefig(str(tacticHash) + "_OneStepBackward.svg")
plt.show()

In [ ]:
data ={}
for tact2, dist in tacticData[tacticHash]['twoNextCount'].items():
  for tact3, count in dist.items():
    tact2name = tacticData[tact2]['text'][0] if tact2 in tacticData else "Done"
    tact3name = tacticData[tact3]['text'][0] if tact3 in tacticData else "Done"
    data.update({(tact2name,tact3name):count})
totaly = 0
for v in data.values():
  totaly += v
plt.pie(data.values(), labels=[k if (v/float(totaly) > 0.02) else '' for k,v  in data.items()])
plt.title("Distribution of pairs of tactics used after " + str(tacticData[tacticHash]['text'][0]))
plt.savefig(str(tacticHash) + "_TwoStepForward.svg")
plt.show()

In [ ]:
sns.histplot(tacticData[tacticHash]['displacements'],log_scale=False)
plt.xlabel("Displacement by " + tacticData[tacticHash]['text'][0])
plt.savefig(str(tacticHash) + "_Displacements.svg")
plt.show()